# Foundational: Dataset-Level ML Lineage with DVC and MLflow on Amazon SageMaker AI

This notebook demonstrates how to build traceable ML workflows using [DVC](https://dvc.org/) for data versioning and [fully managed MLflow on Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) for experiment tracking. You'll learn how to:

- Version image datasets with DVC and store them in Amazon S3
- Track experiments and link models to specific data versions with MLflow
- Run SageMaker AI Processing and Training jobs with full data lineage
- Deploy models from the MLflow Model Registry using the SageMaker v3 SDK (ModelTrainer → Model Package → Endpoint)

This enables **dataset-level lineage** — every model links to a DVC commit hash that points to the exact processed dataset in S3. Given any model, you can reconstruct its training data with `dvc pull`. However, DVC versions the dataset as a unit — there's no structured metadata about which individual records are inside each version. To find out, you'd need to pull the dataset and inspect its contents.

If you need to query which specific records trained a model without reconstructing the dataset (e.g., for opt-out requests or audits), see the [healthcare-compliance](../healthcare-compliance/) variant, which adds a record-level manifest.

### DVC (Data Version Control)

[DVC](https://dvc.org/) extends Git to handle large files and datasets. Instead of storing data in Git, DVC tracks lightweight `.dvc` metafiles while the actual data lives in remote storage like S3.

### MLflow on Amazon SageMaker AI

[MLflow on Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) makes it easier to track experiments and monitor performance of models and AI applications using a single tool. SageMaker AI provides a fully managed MLflow App that integrates natively with SageMaker AI Training jobs.

---

### CIFAR-10 Dataset

We use the [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html), which contains 60,000 32x32 color images in 10 classes:
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

### Data Versioning Story: More Data = Better Model

We simulate a common real-world scenario where you start with limited labeled data and later expand your dataset:

- **v1.0**: Train on 5% of the data (~2,250 training images)
- **v2.0**: Train on 10% of the data (~4,500 training images)

This demonstrates how DVC tracks different dataset versions and MLflow lets you compare model performance as your data grows. You can adjust these fractions to use more data for better accuracy.

## Prerequisites
---

This notebook can run in Amazon SageMaker Studio, a SageMaker AI notebook instance, or locally with AWS credentials configured.

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong>This notebook has been tested using <strong>SageMaker Python SDK version 3.4.1</strong>.
<br> with the following Python runtime versions:
<ul>
<li><strong>Python 3.11</strong></li>
<li><strong>Python 3.12</strong></li>
</ul>
</div>

In [ ]:
# Install dependencies
%pip install --no-cache-dir -r ../requirements.txt

### Restart Your Kernel

In [ ]:
# Restart kernel to get the packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import mlflow
import torch
from importlib.metadata import version
import sys

py_version = f"py{sys.version_info.major}{sys.version_info.minor}"
print(py_version)

PYTORCH_FRAMEWORK_VERSION = {
    "py311": "2.5",
    "py312": "2.6",
}

if py_version not in PYTORCH_FRAMEWORK_VERSION:
    raise RuntimeError(f"No PyTorch inference container for {py_version}. Supported: py311, py312")

pytorch_framework_version = PYTORCH_FRAMEWORK_VERSION[py_version]

print(f"SageMaker SDK version: {version('sagemaker')}")
print(f"MLflow version: {mlflow.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Python version: {py_version}")
print(f"Pytorch framework version: {pytorch_framework_version}")

## Part 1: Configure DVC for Data Versioning
---

We create a subdirectory with a git repository to store DVC metadata. The actual data is stored in Amazon S3.

**Note:** This example uses AWS CodeCommit, but DVC works with any Git provider (GitHub, GitLab, Bitbucket, etc.). Simply replace the `git remote add origin` URL with your repository URL and configure appropriate credentials. The key requirement is that your SageMaker AI execution role (or notebook IAM role) must have permissions to access the Git repository — for CodeCommit, this means `codecommit:GitPull` and `codecommit:GitPush` permissions.

In [ ]:
# Define the DVC repository name
dvc_repo_name = "cifar10-dvc-demo-11"

In [ ]:
%%bash -s "$dvc_repo_name"

repo_name="$1"

# Create CodeCommit repository
aws codecommit create-repository --repository-name ${repo_name} \
    --repository-description "CIFAR-10 image classification with DVC versioning"

account=$(aws sts get-caller-identity --query Account --output text)
region=$(python -c "import boto3;print(boto3.Session().region_name)")
region=${region:-us-east-1}

mkdir -p ${repo_name}
cd ${repo_name}

# Initialize git repo
git init
git branch -M main  
git remote add origin codecommit::${region}://${repo_name}

# Configure git
git config --global user.email "user@domain.com"
git config --global user.name "user"
git config --global credential.helper '!aws codecommit credential-helper $@'
git config --global credential.UseHttpPath true

# Initialize DVC
dvc init
git commit -m 'Initialize DVC'

# Set DVC remote to S3
dvc remote add -d storage s3://sagemaker-${region}-${account}/DEMO-cifar10-dvc
git commit .dvc/config -m "Configure DVC remote"

# Set DVC cache to S3
dvc remote add s3cache s3://sagemaker-${region}-${account}/DEMO-cifar10-dvc/cache
dvc config cache.s3 s3cache
dvc config core.analytics false

git add .dvc/config
git commit -m 'Update DVC config'

git push --set-upstream origin main

## Part 2: Processing and Training with DVC and SageMaker AI
---

We'll run two experiments to show how model performance improves with more data:

- **Experiment 1 (v1.0)**: Train with 5% of the data
- **Experiment 2 (v2.0)**: Train with 10% of the data

Both experiments version the data with DVC and log metrics to MLflow for comparison.

In [ ]:
import boto3
import json
from datetime import datetime

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.image_uris import get_training_image_uri
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingOutput, ProcessingS3Output
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.core import image_uris

# Setup session
sess = Session()
role = get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
account = boto3.client('sts').get_caller_identity()['Account']

dvc_repo_url = f"codecommit::{region}://{dvc_repo_name}"
prefix = 'DEMO-cifar10-dvc'

print(f"Account: {account}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print(f"Role: {role}")

### Setup MLflow Tracking

Create or connect to a SageMaker AI MLflow App for experiment tracking.

> **Estimated time:** Creating a new MLflow App takes ~3-5 minutes.

**Note:** The following cells create an IAM role and MLflow App programmatically. Your notebook's IAM role must have `iam:CreateRole` and `iam:PutRolePolicy` permissions. 

Alternatively, you can create the MLflow App via the [Amazon SageMaker AI console](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-create-tracking-server-studio.html) and skip the role creation cell — just update `mlflow_app_name` to match your existing app.

In [ ]:
import mlflow
import time

sm_client = boto3.client('sagemaker')
iam = boto3.client('iam')

experiment_name = 'demo-cifar10-mlflow-dvc'
mlflow_app_name = 'cifar10-mlflow-app'
mlflow_role_name = 'MLflowAppIAMRole'

# Create IAM role for MLflow if needed
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Least-privilege policy for MLflow App role
# Based on AWS docs: https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup-prerequisites-iam.html
# S3 actions scoped to SageMaker bucket only
mlflow_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:Get*",
                "s3:Put*",
                "s3:List*"
            ],
            "Resource": [
                f"arn:aws:s3:::{bucket}",
                f"arn:aws:s3:::{bucket}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:AddTags",
                "sagemaker:CreateModelPackageGroup",
                "sagemaker:CreateModelPackage",
                "sagemaker:UpdateModelPackage",
                "sagemaker:DescribeModelPackageGroup"
            ],
            "Resource": "*"
        }
    ]
}

try:
    iam.get_role(RoleName=mlflow_role_name)
    print(f"Using existing role: {mlflow_role_name}")
except iam.exceptions.NoSuchEntityException:
    print(f"Creating IAM role: {mlflow_role_name}")
    iam.create_role(
        RoleName=mlflow_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='IAM role for MLflow App'
    )
    iam.put_role_policy(
        RoleName=mlflow_role_name,
        PolicyName='MLflowAppAccess',
        PolicyDocument=json.dumps(mlflow_policy)
    )
    # Wait for IAM role to propagate before using it
    print("Waiting for IAM role to propagate...")
    time.sleep(10)

mlflow_role_arn = f"arn:aws:iam::{account}:role/{mlflow_role_name}"
print(f"MLflow Role ARN: {mlflow_role_arn}")

In [ ]:
# Check for existing MLflow App by name, or create new one.
# AutoModelRegistrationEnabled makes every mlflow.register_model() call also create a
# Model Package in the SageMaker Model Registry (used for deployment in Part 3).
model_registration_mode = 'AutoModelRegistrationEnabled'

def wait_for_mlflow_app(arn):
    while True:
        app = sm_client.describe_mlflow_app(Arn=arn)
        if app['Status'] in ['Created', 'Updated']:
            return app
        if app['Status'] in ['CreateFailed', 'UpdateFailed', 'Deleted']:
            raise RuntimeError(f"MLflow App is in status {app['Status']}")
        print(f"Status: {app['Status']}... waiting")
        time.sleep(30)

apps = sm_client.list_mlflow_apps().get('Summaries', [])
mlflow_app = next((a for a in apps if a['Name'] == mlflow_app_name), None)

if mlflow_app:
    print(f"Using existing MLflow App: {mlflow_app['Name']}")
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app['Arn'])
    if mlflow_app.get('ModelRegistrationMode') != model_registration_mode:
        print(f"Switching ModelRegistrationMode to {model_registration_mode}...")
        sm_client.update_mlflow_app(Arn=mlflow_app['Arn'], ModelRegistrationMode=model_registration_mode)
        mlflow_app = wait_for_mlflow_app(mlflow_app['Arn'])
else:
    print(f"Creating MLflow App: {mlflow_app_name}...")
    response = sm_client.create_mlflow_app(
        Name=mlflow_app_name,
        ArtifactStoreUri=f's3://{bucket}',
        RoleArn=mlflow_role_arn,
        ModelRegistrationMode=model_registration_mode,
    )
    mlflow_app = wait_for_mlflow_app(response['Arn'])

mlflow_app_arn = mlflow_app['Arn']
print(f"MLflow App ARN: {mlflow_app_arn}")
print(f"Model registration mode: {mlflow_app['ModelRegistrationMode']}")


### Experiment 1: Train with 5% of Data (v1.0)
---

First, we process CIFAR-10 using only 5% of the training data. This simulates having limited labeled data when starting a project.

In [ ]:
# Define version for experiment 1
data_version_v1 = "v1.0"
data_fraction_v1 = 0.05  # 5% of training data

timestamp = datetime.now().strftime("%m-%d-%y_%H%M")  # 01-28-26_1427
pipeline_run_id_v1 = f"{data_version_v1}-{timestamp}"

print(f"Pipeline run ID: {pipeline_run_id_v1}")
print(f"Data fraction: {data_fraction_v1} ({int(data_fraction_v1 * 100)}% of training data)")

#### Run Processing Job (v1.0)

> **Estimated time:** ~4-5 minutes

In [ ]:
# Get PyTorch image for processing
processing_image = get_training_image_uri(
    region=region,
    framework="pytorch",
    framework_version=pytorch_framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
)

processor_v1 = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    env={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "PIPELINE_RUN_ID": pipeline_run_id_v1,
    }
)

print(f"Processing image: {processing_image}")

In [ ]:
%%time

processor_v1.run(
    code="preprocessing_foundational.py",
    source_dir="../source_dir",
    arguments=[
        "--data-fraction", str(data_fraction_v1),
        "--data-version", data_version_v1,
        "--val-split", "0.1"
    ],
    wait=True
)

#### Run Training Job (v1.0)

> **Estimated time:** ~4-5 minutes. Includes instance provisioning, DVC pull of the versioned dataset, and training MobileNetV3-Small for 5 epochs.

In [ ]:
# Get PyTorch training image
training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version=pytorch_framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
    image_scope="training"
)

print(f"Training image: {training_image}")

In [ ]:
# Common model name for both versions (same architecture)
registered_model_name = "CIFAR10-MobileNetV3"

model_trainer_v1 = ModelTrainer(
    sagemaker_session=sess,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="../source_dir",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    base_job_name="cifar10-mobilenet-v1",
    hyperparameters={
        "epochs": 5,
        "batch_size": 256,
        "learning_rate": 0.001,
    },
    environment={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "DATA_VERSION": data_version_v1,
        "PIPELINE_RUN_ID": pipeline_run_id_v1,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "MLFLOW_REGISTERED_MODEL_NAME": registered_model_name,
    }
)

In [ ]:
%%time

model_trainer_v1.train()

### Experiment 2: Train with 10% of Data (v2.0)
---

Now we double the dataset size. This simulates having collected more labeled data over time. We expect to see improved accuracy compared to v1.0.

> **Estimated time:** ~8-10 minutes total for processing and training.

In [ ]:
# Define version for experiment 2
data_version_v2 = "v2.0"
data_fraction_v2 = 0.1  # 10% of training data

timestamp = datetime.now().strftime("%m-%d-%y_%H%M")  # 01-28-26_1427
pipeline_run_id_v2 = f"{data_version_v2}-{timestamp}"

print(f"Pipeline run ID: {pipeline_run_id_v2}")
print(f"Data fraction: {data_fraction_v2} ({int(data_fraction_v2 * 100)}% of training data)")

In [ ]:
processor_v2 = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    env={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "PIPELINE_RUN_ID": pipeline_run_id_v2,
    }
)

In [ ]:
%%time

processor_v2.run(
    code="preprocessing_foundational.py",
    source_dir="../source_dir",
    arguments=[
        "--data-fraction", str(data_fraction_v2),
        "--data-version", data_version_v2,
        "--val-split", "0.1"
    ],
    wait=True
)

In [ ]:
model_trainer_v2 = ModelTrainer(
    sagemaker_session=sess,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="../source_dir",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    base_job_name="cifar10-mobilenet-v2",
    hyperparameters={
        "epochs": 5,
        "batch_size": 256,
        "learning_rate": 0.001,
    },
    environment={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "DATA_VERSION": data_version_v2,
        "PIPELINE_RUN_ID": pipeline_run_id_v2,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "MLFLOW_REGISTERED_MODEL_NAME": registered_model_name,
    }
)

In [ ]:
%%time

model_trainer_v2.train()

### Compare Experiments in MLflow
---

Now compare the two training runs in the MLflow UI. The cells below create a [presigned URL](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-launch-ui.html) that logs you into the MLflow App, then open deep links straight into the experiment, the run comparison view, and the metric chart.

> If your browser blocks the pop-ups, allow them for this site or paste the printed URLs into a new tab. The presigned URL can only be redeemed once and expires after 60 seconds, so re-run the first cell to get a fresh one.

You should see:
- v1.0 (5% data): Lower accuracy
- v2.0 (10% data): Higher accuracy

Each run is linked to its exact data version via the `data_version` and `data_git_commit_id` parameters.

![MLflow Experiment Comparison](../img/mlflow_experiment.png)


In [ ]:
import json
import urllib.parse
from IPython.display import Javascript, display

mlflow.set_tracking_uri(mlflow_app_arn)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# The two training runs of this notebook (v1.0 and v2.0), oldest first
training_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string='tags.stage = "training"',
    order_by=["attributes.start_time DESC"],
    max_results=2,
    output_format="list",
)[::-1]
run_ids = [r.info.run_id for r in training_runs]
for r in training_runs:
    print(f"{r.info.run_name:35} run_id={r.info.run_id}  data_version={r.data.params.get('data_version')}")

# A presigned URL logs you into the MLflow UI. It is single-use and expires quickly, so
# open it first (the cell below does), after which the deep links open in the same session.
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_app_arn,
    ExpiresInSeconds=60,
    SessionExpirationDurationInSeconds=1800,
)["AuthorizedUrl"]
mlflow_ui = presigned_url.split("/auth")[0]

# Deep links into the MLflow UI (same URL scheme the UI itself uses)
runs_param = urllib.parse.quote(json.dumps(run_ids))
experiments_param = urllib.parse.quote(json.dumps([experiment_id]))

mlflow_links = {
    "experiment": f"{mlflow_ui}/#/experiments/{experiment_id}",
    "compare_runs": f"{mlflow_ui}/#/compare-runs?runs={runs_param}&experiments={experiments_param}",
    "val_accuracy_chart": f"{mlflow_ui}/#/metric?runs={runs_param}&metric=%22val_accuracy%22&experiments={experiments_param}",
    "latest_run": f"{mlflow_ui}/#/experiments/{experiment_id}/runs/{run_ids[-1]}",
    "registered_models": f"{mlflow_ui}/#/models",
}
print()
for name, url in mlflow_links.items():
    print(f"{name:20} {url}")


In [ ]:
# 1) Log in: open the presigned URL (you can close the tab it opens once the UI has loaded)
display(Javascript('window.open("{}");'.format(presigned_url)))


In [ ]:
# 2) Side-by-side comparison of the v1.0 and v2.0 training runs (parameters, metrics, artifacts)
display(Javascript('window.open("{}");'.format(mlflow_links["compare_runs"])))


In [ ]:
# 3) Validation accuracy of both runs on one chart, and the latest run's detail page
display(Javascript('window.open("{}");'.format(mlflow_links["val_accuracy_chart"])))
display(Javascript('window.open("{}");'.format(mlflow_links["latest_run"])))


### Training Run Details
---

Click into any run to see detailed metrics, parameters, and artifacts. Key information includes:
- Training/validation loss curves over epochs
- Hyperparameters used (learning rate, batch size, epochs)
- DVC data version and Git commit linking the exact dataset
- `sagemaker.training_job_name` / `sagemaker.training_job_arn` tags (also set on the logged model) linking the run to the SageMaker Training job that produced it; `mlflow.source.name` points at the job's console page

![MLflow Training Run Details](../img/mlflow_training_run.png)

### Logged Models
---

Each training run logs its model as an MLflow 3 **logged model** (visible under the run's *Models* tab). Registration to the MLflow Model Registry happens in Part 3, once the model carries everything needed to deploy it:
- Version history of all registered models
- Direct links to the training run and data version that produced each model
- With the MLflow App in `AutoModelRegistrationEnabled` mode, a matching Model Package in the SageMaker Model Registry

![MLflow Registered Model](../img/mlflow_registered_model.png)


## Part 3: Register and Deploy the Model from the MLflow Model Registry
---

Deploy the latest model (v2.0, trained on more data) to a SageMaker AI endpoint, using the MLflow artifact store as the single source of truth for everything the endpoint needs:

1. Resolve the MLflow 3 **logged model** produced by the latest training run
2. Log a `code/inference.py` into the logged model's artifact store with `MlflowClient.log_model_artifacts()`
3. Log a SageMaker **inference specification** (container image, model data location, environment) on the logged model with the `sagemaker-mlflow` plugin
4. Validate the inference code locally, against the exact artifacts the endpoint will download
5. Call `mlflow.register_model()`. Because the MLflow App runs with `AutoModelRegistrationEnabled`, this single call also creates a **Model Package** in the SageMaker Model Registry, and because the specification was logged *first*, that package is created already deployable
6. Approve the Model Package and deploy it with `Model` → `EndpointConfig` → `Endpoint`

```
MLflow artifact store ◀────────── serves directly from ──────────┐
        │  log_model_artifacts() + log_inference_specification()  │
        ▼                                                        │
MLflow ──register──▶ auto-sync ──▶ Model Package (deployable) ──▶ Endpoint
```

No repacking and no artifact copies: the `S3Prefix` `ModelDataSource` downloads the logged model directory as-is to `/opt/ml/model` on the endpoint, so the PyTorch inference container finds `code/inference.py` via `SAGEMAKER_SUBMIT_DIRECTORY` and `model_fn` loads `data/model.pth`.

<div class="alert alert-warning">
<b>Order matters:</b> the specification must be logged <b>before</b> <code>mlflow.register_model()</code>. The auto-sync copies the specification onto the Model Package at registration time. This is why <code>train.py</code> only <i>logs</i> the model and leaves registration to this notebook.
</div>

> **Estimated time:** ~4-5 minutes for endpoint deployment.


### Resolve the logged model of the latest training run

In [ ]:
import os
import tempfile
import importlib.util
from mlflow import MlflowClient
import sagemaker_mlflow

mlflow.set_tracking_uri(mlflow_app_arn)
mlflow_client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# Latest training run (the v2.0 run), then the MLflow 3 logged model it produced. The
# logged model carries the model_id and the S3 artifact_location the inference
# specification will point at.
last_training_run = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string='tags.stage = "training"',
    order_by=["attributes.start_time DESC"],
    max_results=1,
    output_format="list",
)[0]

logged_model = mlflow_client.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"source_run_id = '{last_training_run.info.run_id}'",
    max_results=1,
)[0]

data_version = last_training_run.data.params.get("data_version", "unknown")
data_git_commit_id = last_training_run.data.params.get("data_git_commit_id", "unknown")

print(f"Training run:       {last_training_run.info.run_name} ({last_training_run.info.run_id})")
print(f"Logged model id:    {logged_model.model_id}")
print(f"Artifact location:  {logged_model.artifact_location}")
print(f"Data version:       {data_version} (DVC commit {data_git_commit_id[:12]})")


### Log an `inference.py` into the model's artifact store

The PyTorch inference container (TorchServe + the SageMaker inference toolkit) loads the script named by `SAGEMAKER_PROGRAM` from `SAGEMAKER_SUBMIT_DIRECTORY` and dispatches requests to `model_fn` / `input_fn` / `predict_fn` / `output_fn`.

The `S3Prefix` `ModelDataSource` downloads *everything* under the logged model's artifact location to `/opt/ml/model`, so a file logged at `code/inference.py` arrives at `/opt/ml/model/code/inference.py`, and the MLflow model files (`MLmodel`, `data/model.pth`) land at `/opt/ml/model`.

`model_fn` loads `data/model.pth` directly with `torch.load`. The pickled `nn.Module` needs only `torch`/`torchvision`, which the container already ships, so no `requirements.txt` is installed at startup. The preprocessing in `input_fn` mirrors the validation transform in `source_dir/train.py` (128x128, ImageNet normalization).

In [ ]:
INFERENCE_SCRIPT = r"""
import io
import json
import os

import torch
from PIL import Image
from torchvision import transforms

# Must match the validation transform in source_dir/train.py
TRANSFORM = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def model_fn(model_dir):
    # The S3Prefix ModelDataSource downloads the whole MLflow model directory
    # (MLmodel, data/model.pth, code/inference.py, ...) to model_dir.
    # Load the pickled nn.Module directly with torch. This keeps mlflow out of
    # the serving container and sidesteps mlflow.pytorch.load_model's symlink
    # path check, which rejects /opt/ml/model on SageMaker endpoints.
    model_path = os.path.join(model_dir, "data", "model.pth")
    model = torch.load(model_path, map_location="cpu", weights_only=False)
    model.eval()
    return model


def input_fn(request_body, request_content_type):
    content_type = (request_content_type or "").split(";")[0].strip()
    if content_type in ("image/jpeg", "image/png", "application/x-image"):
        image = Image.open(io.BytesIO(request_body)).convert("RGB")
        return TRANSFORM(image).unsqueeze(0)
    raise ValueError(f"Unsupported content type: {content_type}")


def predict_fn(input_data, model):
    with torch.no_grad():
        logits = model(input_data)
        return torch.softmax(logits, dim=1)


def output_fn(prediction, accept):
    return json.dumps(prediction.tolist()), "application/json"
"""

# log_model_artifacts preserves the local directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py next to MLmodel and data/.
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    with open(os.path.join(tmp, "code", "inference.py"), "w") as f:
        f.write(INFERENCE_SCRIPT)
    mlflow_client.log_model_artifacts(logged_model.model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")


### Log the inference specification

`sagemaker_mlflow.log_inference_specification()` stores the container image, the model data source, and the environment variables as `sagemaker_inference_specification.json` on the logged model. The instance types listed constrain what the Model Package can be deployed on, so include the type you intend to use.

In [ ]:
from sagemaker.core import image_uris

inference_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version=pytorch_framework_version,
    py_version=py_version,
    instance_type="ml.m5.xlarge",
    image_scope="inference"
)

inference_spec = {
    "Containers": [{
        "Image": inference_image,
        # Serve straight from the MLflow artifact store - no repacking, no copies.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location.rstrip("/") + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the inference toolkit at the script logged above.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            "SAGEMAKER_REGION": region,
        },
    }],
    "SupportedContentTypes": ["image/jpeg", "image/png"],
    "SupportedResponseMIMETypes": ["application/json"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge", "ml.c5.xlarge"],
    "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
}

spec_uri = sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print(f"Inference specification logged at {spec_uri}")

### Validate the inference code locally

Before creating anything in SageMaker, check that the code the endpoint will run actually works. Download the same artifacts the `S3Prefix` `ModelDataSource` will download, import `code/inference.py` the same way the container does, and drive the four handlers with a real image. A failure here is a failure the endpoint would otherwise surface as a `/ping` timeout after several minutes of provisioning.

In [ ]:
import requests
from io import BytesIO
from PIL import Image

# Download a test image (dog from PyTorch examples)
url = 'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg'
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()
image_bytes = response.content

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

img = Image.open(BytesIO(image_bytes))
img.thumbnail((300, 300))
img

In [ ]:
# Download the whole logged-model directory: MLmodel, data/, code/, and the spec JSON
local_model_dir = mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{logged_model.model_id}")
print(f"Model artifacts downloaded to: {local_model_dir}")
print(f"Contents:                      {sorted(os.listdir(local_model_dir))}")

# Import code/inference.py exactly like the serving container does, then call model_fn(model_dir)
script_path = os.path.join(local_model_dir, "code", "inference.py")
spec = importlib.util.spec_from_file_location("user_inference_module", script_path)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

local_model = inference_module.model_fn(local_model_dir)
data = inference_module.input_fn(image_bytes, "image/jpeg")
prediction = inference_module.predict_fn(data, local_model)
body, response_content_type = inference_module.output_fn(prediction, "application/json")

local_probs = json.loads(body)[0]
print(f"\nmodel_fn returned:     {type(local_model).__name__}")
print(f"Input tensor shape:    {tuple(data.shape)}")
print(f"Response content type: {response_content_type}")
print(f"Local prediction:      {class_names[int(np.argmax(local_probs))]} ({max(local_probs):.3f})")

### Register the model, and watch the auto-sync

Now register the logged model in the MLflow Model Registry. The sync to the SageMaker Model Registry runs asynchronously, and it records the resulting Model Package ARN as a `sagemaker.model_package_arn` **tag on the MLflow model version**, which is a much simpler way to find the package than searching the registry.

The auto-synced package has **no** approval status at all, so the cell below sets it to `Approved` and adds the MLflow lineage (model id, run id, data version, DVC commit) as customer metadata. The inference specification is already on the package, so nothing about how it is served has to be mutated after the fact.


In [ ]:
registered_model_version = mlflow.register_model(
    f"models:/{logged_model.model_id}", registered_model_name
)
print(f"Registered in MLflow: {registered_model_version.name} v{registered_model_version.version}")

model_package_arn = None
for _ in range(20):
    mv = mlflow_client.get_model_version(registered_model_version.name, registered_model_version.version)
    model_package_arn = mv.tags.get("sagemaker.model_package_arn")
    if model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)

if not model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within the timeout")
print(f"\nSynced SageMaker Model Package: {model_package_arn}")

sm_client.update_model_package(
    ModelPackageArn=model_package_arn,
    ModelApprovalStatus="Approved",
    CustomerMetadataProperties={
        "mlflow_registered_model": f"{registered_model_version.name}/{registered_model_version.version}",
        "mlflow_model_id": logged_model.model_id,
        "mlflow_run_id": logged_model.source_run_id,
        "data_version": data_version,
        "data_git_commit_id": data_git_commit_id,
    },
)

# The package must carry the specification logged in MLflow; fail loudly if the
# environment did not survive the sync (without SAGEMAKER_PROGRAM there is no inference code).
described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
container = described["InferenceSpecification"]["Containers"][0]
model_package_group_name = described["ModelPackageGroupName"]
expected_env = inference_spec["Containers"][0]["Environment"]
mismatched = {k: (v, container.get("Environment", {}).get(k)) for k, v in expected_env.items()
              if container.get("Environment", {}).get(k) != v}
if mismatched:
    raise RuntimeError(f"Environment did not round-trip through the MLflow sync: {mismatched}")

print(f"Model package group: {model_package_group_name}")
print(f"Status:              {described['ModelPackageStatus']} / {described['ModelApprovalStatus']}")
print(f"Image:               {container['Image']}")
print(f"Model data source:   {container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"Environment:         {container.get('Environment', {})}")


### Deploy the Model Package

Deploying from the registry needs no image, no script, and no artifact URI at the call site: all of that is in the package's `InferenceSpecification`. A `ContainerDefinition` that names the Model Package is enough. The three typed resources map one-to-one to the SageMaker API objects: a `Model` (what to serve), an `EndpointConfig` (how much of it, on what hardware), and an `Endpoint` (the running HTTPS service).

> This endpoint has no authentication of its own. Access is controlled entirely by IAM (`sagemaker:InvokeEndpoint`) and it is not exposed to the public internet. Delete it in the cleanup section when you are done.

In [ ]:
import uuid
from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

unique_id = str(uuid.uuid4())[:8]
endpoint_name = f"cifar10-endpoint-{unique_id}"

deployed_model = Model.create(
    model_name=endpoint_name,
    primary_container=ContainerDefinition(model_package_name=model_package_arn),
    execution_role_arn=role,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=endpoint_name,
            initial_instance_count=1,
            instance_type="ml.m5.xlarge",
        )
    ],
)

endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_name,
)
print(f"Creating endpoint {endpoint_name} from {model_package_arn}")

In [ ]:
# Endpoint.create() returns as soon as the request is accepted, so poll until the
# endpoint reaches a terminal state.
while True:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in {"InService", "Failed"}:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': {desc.get('FailureReason')}"
    )
endpoint = Endpoint.get(endpoint_name=endpoint_name)
print(f"Endpoint {endpoint_name} is InService.")

### Test the Endpoint

In [ ]:
runtime_client = boto3.client('sagemaker-runtime')
response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    Body=image_bytes,
    ContentType='image/jpeg',
    Accept='application/json',
)
prediction = json.loads(response['Body'].read().decode('utf-8'))

probs = prediction[0]
predicted_class = int(np.argmax(probs))

print(f'Predicted: {class_names[predicted_class]} ({probs[predicted_class]:.3f})')
print('All probabilities:')
for name, prob in zip(class_names, probs):
    print(f'  {name}: {prob:.3f}')

## Cleanup
---

Delete resources to avoid ongoing charges.

In [ ]:
import time
# Delete endpoint resources
endpoint.delete()
endpoint_config.delete()
deployed_model.delete()
print("Endpoint resources deleted!")

# Optional: remove the auto-synced Model Package and its group from the SageMaker Model Registry
sm_client.delete_model_package(ModelPackageName=model_package_arn)
time.sleep(5)
sm_client.delete_model_package_group(ModelPackageGroupName=model_package_group_name)


In [ ]:
# Optional: Delete MLflow App
# sm_client.delete_mlflow_app(Arn=mlflow_app_arn)

In [ ]:
# Optional: Delete CodeCommit repository
!aws codecommit delete-repository --repository-name {dvc_repo_name}

## Going Further
---

This demo versions data at the dataset level. Here are some directions to extend it:

- **Increase data fractions** — Try training with 25%, 50%, or 100% of CIFAR-10 (adjust `data_fraction`) and compare accuracy improvements in MLflow across more data versions
- **Reproducibility** — Given any model in MLflow, extract its `data_git_commit_id`, run `git checkout <tag> && dvc pull` to recreate the exact training data. This is the core value of dataset-level lineage: any model can be reproduced from its DVC commit
- **Record-level lineage** — If you need to trace individual records (e.g., "which models used record X's data?"), see the [healthcare-compliance](../healthcare-compliance/) variant, which adds a record-level manifest and audit queries

### Speeding Up Iteration

When running repeated experiments (like the v1.0 → v2.0 flow above), two SageMaker AI features help streamline the process:

- **[SageMaker AI Managed Warm Pools](https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html)** — Keep training instances warm between jobs so back-to-back training runs reuse already-provisioned infrastructure. Add `keep_alive_period_in_seconds` to your `Compute` config to enable it. Note that warm pools apply to training jobs only, not processing jobs.

- **[SageMaker AI Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-overview.html)** — Orchestrate the processing → training → registration workflow as a single, repeatable pipeline instead of running each step manually in a notebook. Pipelines handle step dependencies, pass artifacts between steps automatically, and can be triggered programmatically (e.g., when new data is available and you want to create a new dataset version).